# Part 0: Preliminaries

In [18]:
%pip install --quiet --no-cache-dir \
    "boto3>=1.35.0" \
    "botocore>=1.35.0" \
    "httpx>=0.25.0" \
    requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


Nova Compatibility: boto3 needs to be updated to a version that actually knows what us.amazon.nova-lite-v1:0 is

Portability: If you share this notebook with a colleague or move it to a different SageMaker instance type, that line ensures the environment is set up correctly without manual intervention

Dependencies: beautifulsoup4 and requests are not always in the default base kernels, so you'll need them for web scraping or API calls

In [19]:
import boto3
print(boto3.__version__)

1.42.82


In [20]:
import boto3
import json
import requests
from bs4 import BeautifulSoup
import os

# NOTE: Models on Bedrock are frequently updated/deprecated. 

If 'amazon.nova-micro-v1:0' or 'amazon.nova-lite-v1:0' are no longer available, replace them with whatever current
model your GAs (or an AI/web search) recommend. Be careful, though, about the per-token pricing.

# Part 1: Basic Bedrock Client Setup

In [21]:
# Initialize the Bedrock client. This needs to be done once to connect to the service.
bedrock = boto3.client(service_name='bedrock-runtime', region_name='us-east-1')

# Part 2: Text Summarization Function


In [22]:
def summarize_text(text_to_summarize, model_id='amazon.nova-micro-v1:0'):
    """
    Summarizes the given text using a specified Amazon Bedrock model.
    Uses the Converse API — model-agnostic, so the same code works for
    Nova, Titan, Claude, etc. without changing the request format.

    Args:
        text_to_summarize (str): The input text to be summarized.
        model_id (str): The ID of the Bedrock model to use.

    Returns:
        str: The summarized text, or an error message if the invocation fails.
    """
    # Nova Micro supports up to 128K input tokens.
    # Keeping a conservative character limit to stay well within bounds.
    max_input_chars = 24000

    if len(text_to_summarize) > max_input_chars:
        print(f"Warning: Input text is too long ({len(text_to_summarize)} chars). Truncating to {max_input_chars} chars.")
        text_to_summarize = text_to_summarize[:max_input_chars]

    # Wrap the raw text in an explicit summarization instruction.
    # Without this, the model may elaborate or continue the text instead of condensing it.
    prompt = (
        "Summarize the following text concisely. Only output the summary, nothing else.\n\n"
        f"Text:\n{text_to_summarize}"
    )

    try:
        # Converse API: same request format works across all Bedrock model families
        response = bedrock.converse(
            modelId=model_id,
            messages=[
                {
                    "role": "user",
                    "content": [{"text": prompt}]
                }
            ],
            inferenceConfig={
                "maxTokens": 200,    # Max tokens in the generated summary
                "temperature": 0.5,  # Controls randomness (0=deterministic, 1=creative)
                "topP": 0.9          # Nucleus sampling parameter
            }
        )

        # Extract the assistant's response text
        summarized_text = response['output']['message']['content'][0]['text']
        return summarized_text

    except Exception as e:
        return f"Error invoking model '{model_id}': {e}"

# Part 3: Demo Usage

## Example 1: Summarize a hardcoded paragraph

In [23]:
long_article = """
The Amazon rainforest is the largest tropical rainforest in the world, covering an area of approximately 5.5 million square kilometers (2.1 million square miles) across nine South American countries: Brazil, Peru, Ecuador, Colombia, Venezuela, Bolivia, Guyana, Suriname, and French Guiana. It is renowned for its incredible biodiversity, housing an estimated 10% of the world's known species, including countless insects, plants, birds, mammals, and reptiles. The Amazon River, the largest river by discharge volume, flows through the heart of the forest. The rainforest plays a crucial role in regulating global climate patterns by absorbing vast amounts of carbon dioxide and producing oxygen. Deforestation, primarily driven by agriculture, logging, and mining, poses a significant threat to this vital ecosystem, leading to habitat loss, species extinction, and increased carbon emissions. Conservation efforts are underway to protect the Amazon and its unique natural heritage.
"""

print("\n--- Original Text (Example 1) ---")
print(long_article)
print("\n--- Summarized Text (Example 1: Nova Micro) ---")
summary_nova = summarize_text(long_article, model_id='amazon.nova-micro-v1:0')
print(summary_nova)


--- Original Text (Example 1) ---

The Amazon rainforest is the largest tropical rainforest in the world, covering an area of approximately 5.5 million square kilometers (2.1 million square miles) across nine South American countries: Brazil, Peru, Ecuador, Colombia, Venezuela, Bolivia, Guyana, Suriname, and French Guiana. It is renowned for its incredible biodiversity, housing an estimated 10% of the world's known species, including countless insects, plants, birds, mammals, and reptiles. The Amazon River, the largest river by discharge volume, flows through the heart of the forest. The rainforest plays a crucial role in regulating global climate patterns by absorbing vast amounts of carbon dioxide and producing oxygen. Deforestation, primarily driven by agriculture, logging, and mining, poses a significant threat to this vital ecosystem, leading to habitat loss, species extinction, and increased carbon emissions. Conservation efforts are underway to protect the Amazon and its unique

## Example 2: Summarizing a shorter text with Nova Lite

Nova Lite is slightly more capable (and multimodal) but still budget-friendly.

In [24]:
short_text = "The quick brown fox jumps over the lazy dog. This sentence is a pangram, meaning it uses every letter of the alphabet at least once. Pangrams are often used for testing typefaces or keyboard layouts."
print("\n--- Original Text (Example 2) ---")
print(short_text)
print("\n--- Summarized Text (Example 2: Nova Lite) ---")

summary_nova_lite = summarize_text(short_text, model_id='amazon.nova-lite-v1:0')
print(summary_nova_lite)


--- Original Text (Example 2) ---
The quick brown fox jumps over the lazy dog. This sentence is a pangram, meaning it uses every letter of the alphabet at least once. Pangrams are often used for testing typefaces or keyboard layouts.

--- Summarized Text (Example 2: Nova Lite) ---
The text describes a pangram, a sentence using every letter of the alphabet at least once, often used for testing typefaces or keyboard layouts.


### Note on Short Texts

Very short, information-dense texts may not compress much during summarization. Summarization works best when there's redundancy or extraneous information to remove.

In [25]:
long_article = """
The Amazon rainforest is the largest tropical rainforest in the world, covering an area of approximately 5.5 million square kilometers (2.1 million square miles) across nine South American countries: Brazil, Peru, Ecuador, Colombia, Venezuela, Bolivia, Guyana, Suriname, and French Guiana. It is renowned for its incredible biodiversity, housing an estimated 10% of the world's known species, including countless insects, plants, birds, mammals, and reptiles. The Amazon River, the largest river by discharge volume, flows through the heart of the forest. The rainforest plays a crucial role in regulating global climate patterns by absorbing vast amounts of carbon dioxide and producing oxygen. Deforestation, primarily driven by agriculture, logging, and mining, poses a significant threat to this vital ecosystem, leading to habitat loss, species extinction, and increased carbon emissions. Conservation efforts are underway to protect the Amazon and its unique natural heritage.
"""

print("\n--- Original Text ---")
print(long_article)
print("\n--- Summarized Text (Nova Lite) ---")
summary_nova_lite_long = summarize_text(long_article, model_id='amazon.nova-lite-v1:0')
print(summary_nova_lite_long)


--- Original Text ---

The Amazon rainforest is the largest tropical rainforest in the world, covering an area of approximately 5.5 million square kilometers (2.1 million square miles) across nine South American countries: Brazil, Peru, Ecuador, Colombia, Venezuela, Bolivia, Guyana, Suriname, and French Guiana. It is renowned for its incredible biodiversity, housing an estimated 10% of the world's known species, including countless insects, plants, birds, mammals, and reptiles. The Amazon River, the largest river by discharge volume, flows through the heart of the forest. The rainforest plays a crucial role in regulating global climate patterns by absorbing vast amounts of carbon dioxide and producing oxygen. Deforestation, primarily driven by agriculture, logging, and mining, poses a significant threat to this vital ecosystem, leading to habitat loss, species extinction, and increased carbon emissions. Conservation efforts are underway to protect the Amazon and its unique natural her

In [26]:
print("\n--- Summarized Text (Nova Lite) ---")
print(summary_nova_lite_long)

print("\n--- Summarized Text (Nova Micro) ---")
print(summary_nova)


--- Summarized Text (Nova Lite) ---
The Amazon rainforest, the world's largest tropical rainforest, spans 5.5 million square kilometers across nine South American countries, known for its rich biodiversity and significant role in global climate regulation. However, it faces severe threats from deforestation due to agriculture, logging, and mining, prompting ongoing conservation efforts.

--- Summarized Text (Nova Micro) ---
The Amazon rainforest, spanning 5.5 million square kilometers across nine South American countries, is the world's largest tropical rainforest known for its biodiversity and role in climate regulation. Deforestation threatens its ecosystem, prompting conservation efforts.


# Part 4: News Article Summarizer with User Input

I'd like to summarize this article on CNN: https://www.cnn.com/2025/06/24/politics/doge-fired-workers-rehired 
Tuesday, June 24, 2025

In [27]:
import requests
from bs4 import BeautifulSoup

# --- AWS Bedrock Client and Summarization Function (Assumed from previous steps) ---

AWS_REGION = 'us-east-1' # Specify your AWS region

try:
    # This client is for invoking models
    bedrock_runtime = boto3.client(
        service_name='bedrock-runtime',
        region_name=AWS_REGION
    )
    print(f"Bedrock runtime client initialized for region: {AWS_REGION}")
except Exception as e:
    print(f"Error initializing Bedrock runtime client: {e}")
    print("Please ensure your SageMaker execution role has sufficient permissions (e.g., AmazonBedrockFullAccess)")
    print("Also, confirm that Bedrock is enabled in your chosen region.")
    raise # Re-raise to stop execution if essential client isn't available

Bedrock runtime client initialized for region: us-east-1


## Text Summarization Function with Nova

In [28]:
def summarize_text(text_to_summarize, model_id='amazon.nova-micro-v1:0'):
    """
    Summarizes the given text using a specified Amazon Bedrock model.
    Uses the Converse API, which works across model families.
    Assumes 'bedrock_runtime' client is globally available.

    Args:
        text_to_summarize (str): The input text to be summarized.
        model_id (str): The ID of the Bedrock model to use.

    Returns:
        str: The summarized text, or an error message if the invocation fails.
    """
    max_input_chars = 24000  # Conservative limit for Nova models

    if len(text_to_summarize) > max_input_chars:
        print(f"Warning: Input text is too long ({len(text_to_summarize)} chars). Truncating to {max_input_chars} chars.")
        text_to_summarize = text_to_summarize[:max_input_chars]

    # Wrap the raw text in an explicit summarization instruction.
    prompt = (
        "Summarize the following text concisely. Only output the summary, nothing else.\n\n"
        f"Text:\n{text_to_summarize}"
    )

    try:
        response = bedrock_runtime.converse(
            modelId=model_id,
            messages=[
                {
                    "role": "user",
                    "content": [{"text": prompt}]
                }
            ],
            inferenceConfig={
                "maxTokens": 200,
                "temperature": 0.5,
                "topP": 0.9
            }
        )

        summarized_text = response['output']['message']['content'][0]['text']
        return summarized_text

    except Exception as e:
        return f"Error invoking model '{model_id}': {e}"

## Web Content Extraction Function (Revised for CNN)

In [29]:
def get_article_content(url):
    """
    Fetches and extracts main article text from a given URL,
    specifically targeting common CNN article content structures.
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Try common CNN content container classes
        main_content_div = soup.find('div', class_=lambda x: x and (
            'article__content-container' in x or
            'Article__content' in x or
            'l-container' in x and 'cn__content' in x
        ))
        
        article_text = ""
        if main_content_div:
            # Prioritize <p> tags within the main content div
            paragraphs = main_content_div.find_all('p')
            article_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
            
            # Fallback if paragraphs within the main div yield little text
            if not article_text or len(article_text) < 100:
                article_text = main_content_div.get_text(separator='\n').strip()
                article_text = ' '.join(article_text.split()) # Clean up excess whitespace

        # Broader fallback if no specific main_content_div was found
        if not article_text.strip():
            # Try to find text within common article-level tags
            article_tag = soup.find('article')
            if article_tag:
                article_text = article_tag.get_text(separator='\n').strip()
                article_text = ' '.join(article_text.split())
            else:
                # Last resort: collect all paragraphs on the page (can include noise)
                paragraphs = soup.find_all('p')
                article_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
                article_text = ' '.join(article_text.split())


        if not article_text.strip():
            print("Warning: No significant article content found. The page might have an unexpected structure or be heavily script-driven.")
            return None
        
        return article_text.strip()

    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL: {e}. This could be due to network issues, DNS problems, or the website blocking the request.")
        return None
    except Exception as e:
        print(f"Error parsing content: {e}. The website's HTML structure might have changed unexpectedly.")
        return None

## Main Execution Block for News Article Summarization

In [30]:
# --- Main Execution Block for News Article Summarization ---
if __name__ == "__main__":
    print("\n--- News Article Summarizer via Amazon Bedrock ---")
    print("This script will fetch content from a CNN URL and summarize it.")

    ### Older article
    #cnn_article_url = "https://www.cnn.com/2025/06/24/politics/doge-fired-workers-rehired" 
    
    ### Article at time of recording
    cnn_article_url = "https://www.cnn.com/2026/03/31/americas/latin-america-birth-rates-fall-latam-intl"
    article_url_to_process = cnn_article_url

    if article_url_to_process:
        print(f"Attempting to fetch content from: {article_url_to_process}")
        article_content = get_article_content(article_url_to_process)

        if article_content:
            max_input_length_for_bedrock = 24000

            if len(article_content) > max_input_length_for_bedrock:
                print(f"Original article length: {len(article_content)} characters. Truncating to {max_input_length_for_bedrock} characters.")
                article_content_truncated = article_content[:max_input_length_for_bedrock]
            else:
                article_content_truncated = article_content

            print("\n--- Summarizing Article using Amazon Nova Micro ---")
            summarization_prompt = (
                f"Summarize the following news article. Provide the key events, main arguments, "
                f"and any significant outcomes mentioned. Keep the summary concise and informative.\n\n"
                f"Article:\n{article_content_truncated}"
            )

            summary_result = summarize_text(summarization_prompt, model_id='amazon.nova-micro-v1:0')
            print(summary_result)
        else:
            print("Failed to retrieve or parse article content. Please check the URL and the `get_article_content` function for updates.")
    else:
        print("No URL provided for summarization. Exiting.")


--- News Article Summarizer via Amazon Bedrock ---
This script will fetch content from a CNN URL and summarize it.
Attempting to fetch content from: https://www.cnn.com/2026/03/31/americas/latin-america-birth-rates-fall-latam-intl

--- Summarizing Article using Amazon Nova Micro ---
The article highlights the declining fertility rates in Latin America and the Caribbean, noting that the region now averages 1.8 children per woman, below the replacement level of 2.1. This trend is attributed to a generational shift where motherhood is no longer a presumed role, and teenage pregnancies have significantly decreased due to policies promoting reproductive autonomy and access to contraception. Chile has the lowest fertility rate in Latin America at 1.1 children per woman. The decline in birth rates, coupled with rising life expectancies, leads to aging populations that strain economic growth and public services. Experts caution that pro-natalist policies have only had modest effects, emphasiz

## With Nova Lite

In [31]:
summary_result_lite = summarize_text(summarization_prompt, model_id='amazon.nova-lite-v1:0')
print(summary_result_lite)

Latin America and the Caribbean are experiencing a significant decline in birth rates, with an average of 1.8 children per woman, below the replacement level of 2.1. This trend is attributed to factors such as declining teenage pregnancies, increased access to contraception, higher education levels, and changing societal norms. The decline in birth rates is unevenly distributed across income groups and regions, with lower-income women tending to have more children than higher-income women. The shift is leading to aging populations, which poses challenges for economic growth and social services. However, some see potential benefits, such as increased investment per student due to fewer children. Experts emphasize the need for comprehensive policies that address the complex factors driving declining fertility rates.


## Note on Web Scraping

Web scraping is inherently fragile. Websites frequently update their HTML structures, which can break even the most carefully crafted scrapers. If this function stops working in the future for CNN, it's highly likely that their website's HTML for articles has changed, and you'd need to re-inspect and update the selectors.